In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


# BioCLIP linear probe on CUB attributes


## Setup


In [ ]:
# Answers whether BioCLIP encode morphological attributes as separable directions in
# image embedding space, or only species identity 
#
# CUB-200-2011 because it has per-image attribute labels rather than
# species-level ones, which is the label structure trait conditioning requires. 
!pip install -q open_clip_torch scikit-learn pandas
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
CUDA available: True
Device: NVIDIA A100-SXM4-40GB


## Get CUB


In [ ]:

import os

CUB_ROOT = "/content/CUB_200_2011"
CUB_URL = ("https://data.caltech.edu/records/65de6-vp158/"
           "files/CUB_200_2011.tgz")

if not os.path.exists(os.path.join(CUB_ROOT, "images.txt")):
    os.system(f"wget -q --show-progress {CUB_URL} -O /content/cub.tgz")
    os.system("tar -xzf /content/cub.tgz -C /content/")

print("CUB present:", os.path.exists(os.path.join(CUB_ROOT, "images.txt")))


CUB present: True


In [3]:
# Option B: mount Drive and point at an uploaded copy instead.
# from google.colab import drive
# drive.mount('/content/drive')
# CUB_ROOT = str(ROOT / "data/raw/CUB_200_2011")
# print("CUB present:", os.path.exists(os.path.join(CUB_ROOT, "images.txt")))


In [5]:
!ls /content/*.txt
!ls /content/CUB_200_2011/attributes/


/content/attributes.txt
certainties.txt			       image_attribute_labels.txt
class_attribute_labels_continuous.txt


## Metadata


In [ ]:
# 312 binary attributes named like has_bill_shape::dagger. collapse each
# group into one multi-class target by taking the value marked present
# with enough certainty. zero or several present, drop that group only.This is the initial test run before trait labels and is verifiable without acess to the plant-specific data used for the study
import pandas as pd
import numpy as np

def _find(root, name):
    """CUB ships attributes.txt beside CUB_200_2011/, not inside it."""
    parent = os.path.dirname(root.rstrip("/"))
    candidates = [
        os.path.join(root, name),
        os.path.join(parent, name),
        os.path.join(parent, os.path.basename(name)),
        os.path.join(root, os.path.basename(name)),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{name} not found. Tried: {candidates}")

def load_cub(root):
    def read(name, cols):
        return pd.read_csv(_find(root, name), sep=r"\s+",
                           names=cols, header=None)

    images = read("images.txt", ["image_id", "path"])
    labels = read("image_class_labels.txt", ["image_id", "class_id"])
    attrs  = read("attributes/attributes.txt", ["attribute_id", "name"])

    img_attrs = pd.read_csv(
        _find(root, "attributes/image_attribute_labels.txt"),
        sep=r"\s+", header=None, usecols=[0, 1, 2, 3],
        names=["image_id", "attribute_id", "is_present", "certainty_id"],
        on_bad_lines="skip",
    )
    return images.merge(labels, on="image_id"), attrs, img_attrs

def build_attribute_groups(attrs, img_attrs, min_certainty=3, min_count=200):
    attrs = attrs.copy()
    attrs[["group", "value"]] = attrs["name"].str.split("::", expand=True)

    confident = img_attrs[(img_attrs.is_present == 1) &
                          (img_attrs.certainty_id >= min_certainty)]
    confident = confident.merge(attrs[["attribute_id", "group", "value"]],
                                on="attribute_id")

    targets = {}
    for group, block in confident.groupby("group"):
        counts = block.groupby("image_id").size()
        block = block[block.image_id.isin(counts[counts == 1].index)]
        if len(block) < min_count or block.value.nunique() < 2:
            continue
        targets[group] = dict(zip(block.image_id, block.value))
    return targets

images, attrs, img_attrs = load_cub(CUB_ROOT)
targets = build_attribute_groups(attrs, img_attrs)

print(f"{len(images)} images, {len(targets)} usable attribute groups")
for g, m in sorted(targets.items())[:10]:
    print(f"  {g:<32} {len(m):>6} images, {len(set(m.values()))} values")


11788 images, 28 usable attribute groups
  has_back_color                     4544 images, 15 values
  has_back_pattern                   8533 images, 4 values
  has_belly_color                    6643 images, 15 values
  has_belly_pattern                  9830 images, 4 values
  has_bill_color                     8301 images, 15 values
  has_bill_length                   10694 images, 3 values
  has_bill_shape                    10660 images, 9 values
  has_breast_color                   6424 images, 15 values
  has_breast_pattern                10177 images, 4 values
  has_crown_color                    7847 images, 15 values


## Embed


In [ ]:
# image tower only. The
# zero-shot route through trait vocabulary is what this bypasses.
# ~12k images, a few minutes on a T4, cached so the probe can re-run.
import open_clip
from PIL import Image

FEATURE_CACHE = "/content/bioclip_cub_features.npz"
MODEL_ID = "hf-hub:imageomics/bioclip-2"   

def embed_images(image_df, root, batch_size=64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, _, preprocess = open_clip.create_model_and_transforms(MODEL_ID)
    model = model.to(device).eval()

    ids, feats = [], []
    paths = list(zip(image_df.image_id, image_df.path))

    for start in range(0, len(paths), batch_size):
        chunk = paths[start:start + batch_size]
        batch = torch.stack([
            preprocess(Image.open(os.path.join(root, "images", p)).convert("RGB"))
            for _, p in chunk
        ]).to(device)

        with torch.no_grad():
            out = model.encode_image(batch)
            out = out / out.norm(dim=-1, keepdim=True)

        ids.extend(i for i, _ in chunk)
        feats.append(out.cpu().numpy())
        print(f"  {min(start + batch_size, len(paths))}/{len(paths)}", end="\r")

    print()
    return np.array(ids), np.concatenate(feats)

if os.path.exists(FEATURE_CACHE):
    cached = np.load(FEATURE_CACHE)
    ids, feats = cached["ids"], cached["feats"]
    print("loaded feats:", feats.shape)
else:
    ids, feats = embed_images(images, CUB_ROOT)
    np.savez(FEATURE_CACHE, ids=ids, feats=feats)
    print("cached feats:", feats.shape)


open_clip_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

  11788/11788
Embedded and cached: (11788, 768)


## Probe


In [8]:
# balanced accuracy so the baseline sits near 1/k, otherwise a high raw
# accuracy can just be a majority-class artefact. five species splits,
# mean and sd: effective n is 200 species, not 12k images.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit

def probe_group(X, y, groups, seed=0):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    clf = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])

    majority = pd.Series(y[train_idx]).mode()[0]
    baseline = balanced_accuracy_score(y[test_idx],
                                       np.full(len(test_idx), majority))
    return (balanced_accuracy_score(y[test_idx], pred), baseline,
            len(y), len(np.unique(y)))

index   = {int(i): n for n, i in enumerate(ids)}
species = dict(zip(images.image_id, images.class_id))
SEEDS   = 5

rows = []
for group, mapping in sorted(targets.items()):
    usable = [i for i in mapping if i in index]
    if len(usable) < 200:
        continue

    X = feats[[index[i] for i in usable]]
    y = np.array([mapping[i] for i in usable])
    g = np.array([species[i] for i in usable])

    scores = [probe_group(X, y, g, seed=s) for s in range(SEEDS)]
    acc  = np.mean([s[0] for s in scores])
    sd   = np.std([s[0] for s in scores])
    base = np.mean([s[1] for s in scores])

    rows.append({
        "attribute": group.replace("has_", ""),
        "n": scores[0][2],
        "classes": scores[0][3],
        "probe": round(acc, 3),
        "sd": round(sd, 3),
        "baseline": round(base, 3),
        "lift": round(acc - base, 3),
    })
    print(f"  {group:<32} {acc:.3f} (+/- {sd:.3f})")

table = pd.DataFrame(rows).sort_values("lift", ascending=False)
table


  has_back_color                   0.321 (+/- 0.050)
  has_back_pattern                 0.485 (+/- 0.028)


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  has_belly_color                  0.349 (+/- 0.029)
  has_belly_pattern                0.472 (+/- 0.023)


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  has_bill_color                   0.212 (+/- 0.025)
  has_bill_length                  0.676 (+/- 0.025)
  has_bill_shape                   0.533 (+/- 0.023)
  has_breast_color                 0.350 (+/- 0.019)
  has_breast_pattern               0.496 (+/- 0.018)
  has_crown_color                  0.368 (+/- 0.018)


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  has_eye_color                    0.169 (+/- 0.014)
  has_forehead_color               0.361 (+/- 0.020)
  has_head_pattern                 0.286 (+/- 0.016)
  has_leg_color                    0.187 (+/- 0.015)
  has_nape_color                   0.366 (+/- 0.021)
  has_primary_color                0.371 (+/- 0.032)
  has_shape                        0.268 (+/- 0.020)
  has_size                         0.439 (+/- 0.026)
  has_tail_pattern                 0.453 (+/- 0.024)
  has_tail_shape                   0.277 (+/- 0.024)
  has_throat_color                 0.356 (+/- 0.013)


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  has_under_tail_color             0.264 (+/- 0.038)
  has_underparts_color             0.372 (+/- 0.039)
  has_upper_tail_color             0.278 (+/- 0.032)
  has_upperparts_color             0.325 (+/- 0.022)
  has_wing_color                   0.271 (+/- 0.036)
  has_wing_pattern                 0.517 (+/- 0.022)
  has_wing_shape                   0.305 (+/- 0.020)


,attribute,n,classes,probe,sd,baseline,lift
6,bill_shape,10660,9,0.533,0.023,0.111,0.422
5,bill_length,10694,3,0.676,0.025,0.333,0.343
22,underparts_color,5954,15,0.372,0.039,0.067,0.305
15,primary_color,4987,15,0.371,0.032,0.067,0.304
9,crown_color,7847,15,0.368,0.018,0.067,0.301
14,nape_color,6576,15,0.366,0.021,0.067,0.300
11,forehead_color,7921,15,0.361,0.020,0.067,0.295
20,throat_color,8201,15,0.356,0.013,0.067,0.290
7,breast_color,6424,15,0.350,0.019,0.067,0.283
2,belly_color,6643,15,0.349,0.029,0.069,0.280


## Result


In [ ]:
# lift near zero: not linearly decodable, the representation doesn't
# separate it. lift well above zero means the information is in the image tower,
# meaning a zero-shot failure on the same attribute was the text bridge, not 
# representation, and a supervised probe has the potential for utilisation as an automated scorer.

table.to_csv("/content/bioclip_cub_probe_results.csv", index=False)

decodable   = table[table.lift > 0.10]
undecodable = table[table.lift <= 0.05]

print(f"decodable (lift > 0.10): {len(decodable)}/{len(table)}")
print(f"Not decodable      (lift <= 0.05): {len(undecodable)}/{len(table)}")
print("\nStrongest:")
print(decodable.head(5).to_string(index=False))
print("\nWeakest:")
print(table.tail(5).to_string(index=False))


Linearly decodable (lift > 0.10):  27/28
Not decodable      (lift <= 0.05): 0/28

Strongest:
       attribute     n  classes  probe    sd  baseline  lift
      bill_shape 10660        9  0.533 0.023     0.111 0.422
     bill_length 10694        3  0.676 0.025     0.333 0.343
underparts_color  5954       15  0.372 0.039     0.067 0.305
   primary_color  4987       15  0.371 0.032     0.067 0.304
     crown_color  7847       15  0.368 0.018     0.067 0.301

Weakest:
 attribute    n  classes  probe    sd  baseline  lift
bill_color 8301       15  0.212 0.025     0.068 0.145
 leg_color 7679       15  0.187 0.015     0.067 0.120
tail_shape 7578        6  0.277 0.024     0.167 0.110
wing_shape 5423        5  0.305 0.020     0.200 0.105
 eye_color 9935       14  0.169 0.014     0.076 0.093
